# 04B Real CIF → Descriptors → ML Table

> 🔵 **Level B · Recommended**

`real COF CIF → parse → QC → descriptors → COF_ID → target table → ML` using CURATED-COFs structures.


## Scope of this exercise
Density comes from real CIFs, but this is a pipeline exercise, not an adsorption model. Density equals cell mass divided by volume and can be computed exactly from a complete structure. Excluding atom count and volume does not prove independence: lattice parameters still encode volume. Use an independently sourced property for a scientific prediction task.


## 1. Complete materials-ML data flow

```text
CIF files → pymatgen Structure → descriptors → feature table
                                      ↓
                                    COF_ID
                                      ↓
                                target table
                                      ↓
                              merge / quality control
                                      ↓
                           train / test / model / evaluation
```

`COF_ID` is the key concept. Structures, descriptors, simulations, and experiments often live in separate files. A stable identifier makes the mapping reproducible.


In [ ]:
!pip -q install pymatgen scikit-learn
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymatgen.core import Structure
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
META_URL = 'https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cof-frameworks.csv'
meta = pd.read_csv(META_URL).rename(columns={'CURATED-COFs ID': 'COF_ID'})
display(meta.head())
N_COF = 80
subset = meta.head(N_COF).copy()


## 2. CIF → descriptors

This baseline extracts lattice parameters, cell volume, atom counts, element fractions, density, and a simple cell-shape descriptor. **LCD, PLD, accessible surface area, void fraction, and pore volume are not reliably produced by this simple pymatgen block.** They normally require dedicated pore-analysis software such as Zeo++ or PoreBlazer.


In [ ]:
ELEMENTS = ['H','B','C','N','O','F','S']
def structure_record(cof_id, structure):
    comp = structure.composition
    rec = {'COF_ID':cof_id,'a_A':structure.lattice.a,'b_A':structure.lattice.b,'c_A':structure.lattice.c,'alpha_deg':structure.lattice.alpha,'beta_deg':structure.lattice.beta,'gamma_deg':structure.lattice.gamma,'cell_volume_A3':structure.volume,'n_atoms':len(structure),'n_elements':len(comp.elements),'density_g_cm3':float(structure.density)}
    abc=np.array([rec['a_A'],rec['b_A'],rec['c_A']]); rec['cell_anisotropy']=abc.max()/abc.min()
    for el in ELEMENTS: rec[f'frac_{el}']=comp.get_atomic_fraction(el)
    return rec
records, failed = [], []
for cof_id in subset['COF_ID']:
    url=f'https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cifs/{cof_id}.cif'
    try:
        r=requests.get(url,timeout=20); r.raise_for_status(); s=Structure.from_str(r.text,fmt='cif'); records.append(structure_record(cof_id,s))
    except Exception as exc: failed.append((cof_id,type(exc).__name__ + ": " + str(exc)))
features_all=pd.DataFrame(records)
print('parsed =',len(features_all),'failed =',len(failed)); display(features_all.head())

if len(features_all) < 10:
    raise RuntimeError(f"Too few parsed structures: {len(features_all)}; inspect failures: {failed[:5]}")
display(pd.DataFrame(failed, columns=["COF_ID", "error"]).head(10))


## 3. Quality control before ML

Real CIF files may contain guests/solvent, missing H atoms, disorder, partial occupancies, unusual cells, duplicates, or questionable stacking. Do not hide such issues with a blind `dropna()`.


In [ ]:
qc_cols=['a_A','b_A','c_A','cell_volume_A3','n_atoms','density_g_cm3','cell_anisotropy']
display(features_all[qc_cols].describe().T)
display(features_all.isna().sum().sort_values(ascending=False).head(10))


## 4. Separate feature and target tables, then merge by COF_ID

In a research project the target table may come from GCMC, experiments, or literature. The `merge()` logic remains the same.


In [ ]:
target_table=features_all[['COF_ID','density_g_cm3']].copy()
feature_cols=['a_A','b_A','c_A','alpha_deg','beta_deg','gamma_deg','cell_anisotropy','n_elements','frac_H','frac_B','frac_C','frac_N','frac_O','frac_F','frac_S']
feature_table=features_all[['COF_ID']+feature_cols].copy()
dataset=feature_table.merge(target_table,on='COF_ID',validate='one_to_one')
display(dataset.head()); print('merged rows =',len(dataset))


Excluding direct density components makes a weaker teaching baseline; it is not a proof that the task is free of shortcuts. Lattice parameters determine cell volume. Compare with the exact mass/volume calculation before interpreting model scores.


In [ ]:
X=dataset[feature_cols]; y=dataset['density_g_cm3']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)
model=RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)
model.fit(X_train,y_train); pred=model.predict(X_test)
print('MAE  =',mean_absolute_error(y_test,pred))
print('RMSE =',mean_squared_error(y_test,pred)**0.5)
print('R2   =',r2_score(y_test,pred))


In [ ]:
plt.figure(figsize=(5,5)); plt.scatter(y_test,pred)
lo=min(y_test.min(),pred.min()); hi=max(y_test.max(),pred.max()); plt.plot([lo,hi],[lo,hi],'--')
plt.xlabel('True density (g/cm³)'); plt.ylabel('Predicted density (g/cm³)'); plt.title('Real-CIF baseline'); plt.show()
importance=pd.Series(model.feature_importances_,index=feature_cols).sort_values(ascending=False); display(importance.to_frame('importance').head(10))


## 5. What comes next?

A research-grade COF workflow should add pore descriptors, chemical-environment descriptors, a rigorously defined target at consistent conditions, structure/family-aware splitting, interpretation, and candidate screening. 04C maps these ideas onto a published COF CO₂ high-throughput-screening workflow.


## Exercises

1. Increase `N_COF` to 150 and record the parsing failure rate.
2. Add `cell_volume_A3` and `n_atoms`; explain the performance change and whether it is a shortcut.
3. Plot lattice, density, and element-fraction distributions.
4. Return the five highest- and lowest-density COFs with IDs and names.
5. Design the columns of a `pore_features.csv` produced by Zeo++.
6. Explain why `COF_ID` acts as the primary key connecting structures, features, and targets.

### Completion criterion
You can generate an ML table from real CIF files and explain which descriptors come directly from CIF, which require separate pore calculations, where the target comes from, and how all tables are aligned.


## Export traceable tables
Save descriptors, targets and failures together. IDs from another database require a verified crosswalk; do not assume that similarly named materials are identical.


In [ ]:
from pathlib import Path
output_dir = Path('cof_cif_outputs')
output_dir.mkdir(exist_ok=True)
feature_table.to_csv(output_dir/'features.csv', index=False)
target_table.to_csv(output_dir/'targets.csv', index=False)
pd.DataFrame(failed, columns=['COF_ID','error']).to_csv(output_dir/'failed.csv', index=False)
print(output_dir.resolve())


## Sources and further reading
[Dataset contracts / 数据使用约定](../../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
